<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day19_practice2_ViT_%EC%A1%B0%EB%A6%BD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ViT 조립과 학습

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, os, urllib.request
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 3

In [3]:
# 셀 1. 데이터
def load_fmnist(root="./data", train=True):
  tf = transforms.ToTensor() # 이미지(0-255) → 텐서(0-1)
  return datasets.FashionMNIST(root=root, train=train, download=True, transform=tf)

train_loader = DataLoader(load_fmnist(train=True), batch_size=128, shuffle=True)
test_data = load_fmnist(train=False)
test_loader = DataLoader(test_data, batch_size=512)

100%|██████████| 26.4M/26.4M [00:02<00:00, 8.89MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 138kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 2.59MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 15.1MB/s]


In [4]:
# 셀 2.
class MultiHeadAttention(nn.Module):
  def __init__(self, dim, n_heads): # dim = 128, n_heads = 8 헤드 수
    super().__init__()

    assert dim % n_heads == 0
    self.n_heads = n_heads
    self.d_heads = dim // n_heads # d_head=8 헤드 하나가 맡는 차원 수

    self.W_q = nn.Linear(dim, dim, bias=False) # 각 헤드마다 Q,K,V를 만들 학습 행렬
    self.W_k = nn.Linear(dim, dim, bias=False)
    self.W_v = nn.Linear(dim, dim, bias=False)
    self.W_o = nn.Linear(dim, dim, bias=False) # 헤드들 결과를 합친 뒤 섞는 출력층

  def forward(self, x, mask=None):
    B, L, D = x.shape # x:(B,17,64) 배치, cls+패치, 벡터차원
    H, dh = self.n_heads, self.d_heads # H=8 헤드수, dh=16 각헤드 차원수

    # 1. 쪼개기 x:(B,17,64)→(B,17,8,8)→(B,8,17,8)
    Q = self.W_q(x).reshape(B, L, H, dh).transpose(1, 2) # (B,8,17,8) 배치, 멀티헤드수, cls+패치, 벡터차원8
    K = self.W_k(x).reshape(B, L, H, dh).transpose(1, 2) # (B,8,17,8)
    V = self.W_v(x).reshape(B, L, H, dh).transpose(1, 2) # (B,8,17,8)

    # 2. 각자 보기 : 셀프어텐션
    scores = Q @ K.transpose(-2, -1) / math.sqrt(dh) # (B,8,17,8) @ (B,8,8,17) → (B,8,17,17) (B, H, QL, KL) 관련도 점수
    if mask is not None:
      scores = scores.masked_fill(mask, -1e9) # masked_fill(조건, 값) 조건이 True인 위치를 값으로 채운다. (-10억 (-무한대))가 softmax를 통과하면 0이 된다.
    attn = F.softmax(scores, dim=-1) # softmax에서 dim=-1 축을 따라 정규화, (축은 그대로 남아있고, 그 축의 값들을 합=1로) (B,8,17,17)
    self.attn_map = attn # 시각화용 저장 (B,8,17,17)
    out = attn @ V # (B,8,17,8)

    # 3. 이어붙이기 : 헤드 8개(각 8차원)를 다시 원래 64차원 한 줄로 이어붙여 입력과 같은 모양 복원
    out = out.transpose(1,2).reshape(B, L, D) # (B,8H,64L,16벡터) → (B,64L,8H,16벡터) → (B,64L,128벡터)

    # 4. 섞기 : 헤드 8개 결과를 어떻게 조합할지도 학습(W_o)
    return self.W_o(out) # (B,17,64)

class TransformerBlock(nn.Module):
  def __init__(self, dim=64, n_heads=8):
    super().__init__()
    self.attn = MultiHeadAttention(dim, n_heads) # 단어끼리 정보 교환
    self.norm1 = nn.LayerNorm(dim)
    self.ffn = nn.Sequential( # FFN
        nn.Linear(dim, dim * 4), nn.ReLU(), nn.Linear(dim * 4, dim)
    )
    self.norm2 = nn.LayerNorm(dim)

  def forward(self, x, mask=None): # x:h(B,64L,128벡터)
    x = self.norm1(x + self.attn(x, mask)) # ＋× 잔차
    x = self.norm2(x + self.ffn(x))
    return x # (B,64L,128벡터) 입력과 똑같은 모양

In [6]:
# 셀 3. ViT
PATCH, DIM, N_PATCH = 7, 64, 16

class ViT(nn.Module):
  def __init__(self, n_blocks=2, n_heads=8):
    super().__init__()
    self.patch_embed = nn.Linear(PATCH * PATCH, DIM) # 49→64 차원 임베딩
    self.cls = nn.Parameter(torch.zeros(1,1,DIM)) # (1,1,64)요약 토큰
    self.pos = nn.Parameter(torch.zeros(1,N_PATCH + 1, DIM)) # (1,17,64) 위치 임베딩
    self.blocks = nn.ModuleList(
        TransformerBlock(DIM, n_heads) for _ in range(n_blocks)
    )
    self.head = nn.Linear(DIM, 10) # 64→10클래스 ([CLS]→분류)

  def to_patches(self, images):
    B = images.shape[0]
    p = images.unfold(2, PATCH, PATCH).unfold(3, PATCH, PATCH) # (B,1,4,4,7,7)
    return p.reshape(B, N_PATCH, PATCH * PATCH) # (B,16,49) 16패치, 49픽셀

  def forward(self, images): # images:(B,1,28,28)
    x = self.patch_embed(self.to_patches(images)) # (B,1,28,28)→(B,16,49)→(B,16,64)
    x = torch.cat([self.cls.expand(x.size(0), -1, -1), x], 1) # (B,1,64)+(B,16,64)→(B,17,64)
    x = x + self.pos # (B,17,64) 위치 임베딩 더함
    for b in self.blocks:
      x = b(x) # (B,17,64)
    return self.head(x[:, 0]) # (B,64) cls 자리만 → (B,10) CLS가 어텐션으로 16패치를 다 모은 요약 그 벡터 하나로 분류

vit = ViT().to(device)
n_vit = sum(p.numel() for p in vit.parameters())
print(f"ViT 파라미터: {n_vit/1e3:.0f}K")

ViT 파라미터: 104K


In [7]:
# 셀 4. ViT vs CNN
def make_cnn():
  return nn.Sequential(
      nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2), # 14*14
      nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2), #7*7
      nn.Flatten(), nn.Linear(32*7*7, 128), nn.ReLU(), nn.Dropout(0.3),
      nn.Linear(128, 10)
  )
def train_eval(model, name):
  torch.manual_seed(42)
  model = model.to(device)
  opt = torch.optim.Adam(model.parameters(), lr=0.001)
  loss_fn = nn.CrossEntropyLoss()
  for epoch in range(EPOCHS):
    model.train()
    for xb, yb in train_loader:
      xb, yb = xb.to(device), yb.to(device)
      loss = loss_fn(model(xb), yb)
      opt.zero_grad(); loss.backward(); opt.step()
  model.eval(); correct = 0
  with torch.no_grad():
    for xb, yb in test_loader:
      correct += (model(xb.to(device)).argmax(1) == yb.to(device)).sum().item()
  acc = correct / len(test_data)
  print(f"{name}: 테스트 정확도 {acc:.4f}")
  return acc

print(f"[F-MNIST {EPOCHS} epoch]")
acc_cnn = train_eval(make_cnn(), "CNN")
acc_vit = train_eval(vit, "ViT")

# 중간 규모(ImageNet)까진 CNN이 앞서고, 초대규모(JFT-300M, 3억장)에서 ViT가 역전한다

[F-MNIST 3 epoch]
CNN: 테스트 정확도 0.8834
ViT: 테스트 정확도 0.8508
